# 02 | ทำความสะอาดและเตรียมข้อมูล

รักษาหน่วยวิเคราะห์เดิมคือหนึ่งแถวต่อรีวิว คงฟิลด์ต้นทางทั้ง 13 คอลัมน์ และเพิ่มฟิลด์สำหรับรายงานคุณภาพ เวลา และกลุ่มคะแนน ดูความหมายเต็มใน [พจนานุกรมข้อมูล](../docs/data_dictionary.md)

In [ ]:
from pathlib import Path
import hashlib

import pandas as pd

# รองรับการเปิดจากโฟลเดอร์โปรเจกต์หรือจาก notebooks/
working_dir = Path.cwd()
root = working_dir.parent if working_dir.name == "notebooks" else working_dir
if not (root / "README.md").is_file() or not (root / "notebooks").is_dir():
    raise FileNotFoundError("Open the notebook from the portfolio folder or its notebooks/ folder.")

raw_name = "samplehotel - Sheet1.csv"
raw_path = root / "data" / "raw" / raw_name
if not raw_path.is_file():
    raise FileNotFoundError(f"Missing data/raw/{raw_name}. See README.md for setup.")
source_hash = hashlib.sha256(raw_path.read_bytes()).hexdigest()
print(f"Source: {raw_path.relative_to(root)}")
print(f"SHA-256: {source_hash}")

In [ ]:
processed_dir = root / "data" / "processed"
cleaned_path = processed_dir / "cleaned_hotel_reviews.csv"
manifest_path = processed_dir / "source.json"

## ตรวจคะแนน วันที่ และค่าผิดปกติก่อนแก้ไข

คะแนนต้องเป็นจำนวนเต็ม 1–5 และวันที่เขียนต้องแปลงได้ ถ้าพบค่าผิดเงื่อนไข ให้หยุดก่อนเขียนผลเพื่อกลับไปตรวจต้นทาง ตรวจช่วงและความสอดคล้องของฟิลด์ Experience แยกต่างหาก เพราะกราฟเวลายึดวันที่เขียน

In [ ]:
raw = pd.read_csv(raw_path, keep_default_na=False, encoding="utf-8-sig")
reviews = raw.copy()
score = pd.to_numeric(raw["rating"], errors="coerce")
written = pd.to_datetime(raw["clean_written_date"], errors="coerce")
experience_month = pd.to_numeric(raw["experience_month"], errors="coerce")
experience_year = pd.to_numeric(raw["experience_year"], errors="coerce")
raw_text = raw["text"].astype("string")
text_key = raw_text.str.strip().str.replace(r"\s+", " ", regex=True).str.casefold()
invalid_score = ~score.isin(range(1, 6))
invalid_date = written.isna()

display(pd.Series({
    "rows_before": len(raw),
    "invalid_ratings": invalid_score.sum(),
    "unparseable_written_dates": invalid_date.sum(),
    "experience_month_outside_1_12": (~experience_month.isin(range(1, 13))).sum(),
    "experience_year_differs_or_is_missing": experience_year.ne(written.dt.year).sum(),
    "experience_month_differs_or_is_missing": experience_month.ne(written.dt.month).sum(),
    "exact_repeated_rows": raw.duplicated().sum(),
    "normalized_repeated_nonblank_texts": text_key[text_key.ne("")].duplicated().sum(),
}))
if raw.empty or invalid_score.any() or invalid_date.any():
    raise ValueError("Review the empty file, ratings or written dates flagged above before export.")

## จัดการค่าว่างและสร้างฟิลด์

- ตัดช่องว่างหัวท้ายในฟิลด์ชื่อและหมวดหมู่ แทนช่องว่างหรือ `Not Specified` ด้วย `Unknown` สำหรับประเทศ ประเภทนักเดินทาง และช่องทางโดยไม่เดาค่า ส่วนชื่อผู้รีวิวที่ว่างใช้ `Unknown Reviewer`
- เก็บ `title` และ `text` ตามต้นทางเพื่อรักษาบริบท หากข้อความว่างจะเห็นจากความยาวและรายงานคุณภาพ
- เก็บแถวซ้ำไว้เพื่อให้ตรวจสอบต่อได้ เพราะยังไม่มีรหัสรีวิวที่ใช้ยืนยันว่าเป็นรายการเดียวกัน ไฟล์ต้นทางชุดนี้ไม่มีแถวซ้ำ
- คะแนน 1–3 เป็นกลุ่มสำรวจปัญหา คะแนน 4 และ 5 แยกไว้เพื่อเห็นความแตกต่างของสัดส่วน

In [ ]:
for col in ["displayName", "clean_country", "travel_type", "publish_platform", "name", "hotel_location"]:
    values = reviews[col].astype("string").str.strip()
    if col == "displayName":
        values = values.mask(values.eq(""), "Unknown Reviewer")
    elif col in {"clean_country", "travel_type", "publish_platform"}:
        values = values.mask(values.isin(["", "Not Specified"]), "Unknown")
    reviews[col] = values
reviews["rating"] = score.astype("int64")
reviews["clean_written_date"] = written.dt.normalize()
reviews["review_year"] = written.dt.year
reviews["review_month"] = written.dt.month
reviews["review_month_start"] = written.dt.to_period("M").dt.to_timestamp()
reviews["review_char_count"] = raw_text.str.len()
reviews["possible_truncation"] = raw_text.str.len().eq(503) & raw_text.str.endswith("...")
reviews["rating_group"] = score.map({1: "Low (1-3)", 2: "Low (1-3)", 3: "Low (1-3)", 4: "4 stars", 5: "5 stars"})

## ตรวจสอบก่อนและหลัง แล้วส่งออก

เปรียบเทียบจำนวนทุกคะแนนและค่าเฉลี่ยกับต้นทาง ก่อนเขียน CSV แยกต่างหาก ไฟล์ `source.json` ระบุชื่อไฟล์และค่า SHA-256 เพื่อให้ EDA ตรวจได้ว่าผลมาจาก Raw Data ชุดไหน

In [ ]:
import json

rating_check = pd.DataFrame({
    "before": score.value_counts().reindex(range(1, 6), fill_value=0),
    "after": reviews["rating"].value_counts().reindex(range(1, 6), fill_value=0),
})
display(rating_check)
display(pd.Series({
    "rows_before": len(raw),
    "rows_after": len(reviews),
    "rows_removed": len(raw) - len(reviews),
    "mean_before": score.mean(),
    "mean_after": reviews["rating"].mean(),
    "unknown_reviewer": reviews["displayName"].eq("Unknown Reviewer").sum(),
    "unknown_country": reviews["clean_country"].eq("Unknown").sum(),
    "unknown_traveler_type": reviews["travel_type"].eq("Unknown").sum(),
    "blank_review_text": text_key.eq("").sum(),
    "possible_truncation": reviews["possible_truncation"].sum(),
}))
assert len(raw) == len(reviews) and rating_check["before"].equals(rating_check["after"])
assert raw["text"].equals(reviews["text"])
assert hashlib.sha256(raw_path.read_bytes()).hexdigest() == source_hash

processed_dir.mkdir(parents=True, exist_ok=True)
reviews.to_csv(cleaned_path, index=False, encoding="utf-8", date_format="%Y-%m-%d")
manifest = {"source_file": raw_name, "source_sha256": source_hash, "rows": len(reviews)}
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
print(f"Saved {cleaned_path.relative_to(root)}")